# 数字与类别混在一起，怎样预测房价？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{'selected_by_validation': 'boosting', 'validation_mae': {'ridge': 29.377516458812316, 'boosting': 18.43812590797727}, 'test_mae': {'ridge': 28.349006127844724, 'boosting': 17.31136141207576}}


## 1. 补全：独立复算误差
对验证选中的模型重算测试 MAE，确认与 metrics 一致。

In [2]:
manual = float(np.abs(y_test.to_numpy() - predictions[selected].to_numpy()).mean())
assert np.isclose(manual, metrics['test_mae'][selected])
print(manual)

17.31136141207576


## 2. 修改：预测未见区域
创建一套 area 缺失、district=east 的房屋，验证流水线能返回有限值，并解释未知区域的不确定性。

In [3]:
new_house = pd.DataFrame({'area':[np.nan], 'age':[10], 'district':['east']})
value = models[selected].predict(new_house)
assert value.shape == (1,) and np.isfinite(value).all()
print('未见区域预测，仅作机制演示', value)

未见区域预测，仅作机制演示 [404.66038465]


## 3. 迁移：房租预测
构造房租为价格的 0.004 倍加小噪声，在同一训练/验证索引上训练；不要混用房价指标单位。

In [4]:
rent = prices * 0.004 + np.random.default_rng(71).normal(0,0.05,n)
rent_model = make_models()['ridge']
rent_model.fit(houses.loc[X_train.index], rent.loc[X_train.index])
rent_mae = mean_absolute_error(rent.loc[X_val.index], rent_model.predict(houses.loc[X_val.index]))
assert np.isfinite(rent_mae) and rent_mae >= 0
print('房租验证 MAE / 教学单位', rent_mae)

房租验证 MAE / 教学单位 0.12156748871674061
